# ⏱️ POC 6: Reallocation Frequency Optimization across Broad Universe ($M=100$)

**File**: [`research/notebooks/algo-alpha-execution/06_rebalance_frequency_optimization.ipynb`](file:///c:/Users/honza/Desktop/projects/stock-analysis/research/notebooks/algo-alpha-execution/06_rebalance_frequency_optimization.ipynb)  
**Historical Backtest Horizon**: **January 2021 - August 2026 (5.6 Years / 1,414 Daily Trading Sessions)**  
**Broad High-Asymmetry Universe ($M=100$ Stocks)**: 100 liquid US equities across Tech, Semis, Defense, Healthcare, Financials, Consumer Retail, Energy, and Utilities.  

---

### Executive Summary & Quantitative Objective
When tested across a realistic broad universe of 100 stocks (where only a minority are winners and the majority are market-performers or laggards), the sensitivity of reallocation frequency reveals the fundamental interaction between **signal discovery**, **trailing stop holding periods**, and **transaction costs**.

For each reallocation method, this notebook runs an exhaustive daily grid sweep from **$k = 1$ to $120$ trading days** and renders **two diagnostic plots per method**:
1. **Plot 1: Profit vs. Reallocation Frequency Curve**: Continuous curve showing Final Total Return (%) as a function of reallocation frequency ($k \in [1, 120]$ days).
2. **Plot 2: Top 10 Best Frequencies Multi-Curve Time Series**: Cumulative portfolio equity trajectories ($100 starting capital, 2021-2026) for the 10 highest-profit frequencies with right-aligned legends.

---

### Reallocation Methods Evaluated ($M=100$)
1. **Method 1: Unified "One-for-All" Alpha Engine (Flagship)**: Confluence dynamic sizing (5%–20%), trailing ATR stops ($2.5 \cdot \text{ATR}_{14}$), macro volatility spike guard ($2.0\sigma$), and 15 bps friction.
2. **Method 2: Multi-Modal Alpha (Active Top 10, Fixed 10% Cap)**: 10% position caps, 10% stop-loss, inverse-volatility sizing, and 15 bps friction.
3. **Method 3: Baseline Naive XGBoost (Top 10 Active)**: Equal-weight fundamental + TA baseline model with 15 bps friction.
4. **Method 4: Smart Buy & Hold (Top 10 Market-Cap Value-Weighted)**: Point-in-time Top 10 largest stocks by market capitalization, weighted by market value vector $\mathbf{w}_t = \frac{\mathbf{MV}_t}{\sum \mathbf{MV}_t}$ with 15 bps friction.

## 1. Setup, Configuration & Dependencies

In [1]:
import os
import sys
import datetime
from datetime import timedelta
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm.auto import tqdm

# Robust project root discovery
current_dir = os.path.abspath(os.getcwd())
while current_dir and not os.path.exists(os.path.join(current_dir, "src")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

PROJECT_ROOT = current_dir
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import DATA_DIR, INITIAL_CAPITAL

LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "research", "notebooks", "algo-alpha-execution", "data", "fetched")
if not os.path.exists(LOCAL_DATA_DIR):
    LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "fetched")

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Local Data Directory: {LOCAL_DATA_DIR}")
print(f"💰 Initial Capital: ${INITIAL_CAPITAL}")
print(f"🔄 Frequency Grid Range: 1 to 120 Days across M=100 Universe")

📁 Project Root: c:\Users\honza\Desktop\projects\stock-analysis
📁 Local Data Directory: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched
💰 Initial Capital: $100.0
🔄 Frequency Grid Range: 1 to 120 Days across M=100 Universe


## 2. Ingesting Broad 100-Ticker Predictions, Market Prices, ATR & Volatility

In [2]:
def load_sweep_data():
    preds_path = os.path.join(LOCAL_DATA_DIR, "expanded_100tickers_predictions_poc.xlsx")
    df_preds = pd.read_excel(preds_path)
    df_preds['date'] = pd.to_datetime(df_preds['date'])
    all_tickers = sorted(df_preds['ticker'].unique())
    
    unique_tickers = all_tickers + ['SPY']
    min_date = (df_preds['date'].min() - timedelta(days=60)).strftime('%Y-%m-%d')
    max_date = (df_preds['date'].max() + timedelta(days=10)).strftime('%Y-%m-%d')
    
    print(f"📈 Downloading OHLC market data for {len(unique_tickers)} tickers...")
    ohlc = yf.download(unique_tickers, start=min_date, end=max_date, auto_adjust=True, progress=False)
    
    close_p = ohlc['Close']
    high_p = ohlc['High']
    low_p = ohlc['Low']
    
    close_p.index = pd.to_datetime(close_p.index).tz_localize(None)
    high_p.index = pd.to_datetime(high_p.index).tz_localize(None)
    low_p.index = pd.to_datetime(low_p.index).tz_localize(None)
    
    # Compute ATR(14)
    atr_dict = {}
    for t in all_tickers:
        if t in close_p.columns and t in high_p.columns and t in low_p.columns:
            c = close_p[t]
            h = high_p[t]
            l = low_p[t]
            prev_c = c.shift(1)
            tr = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
            atr_dict[t] = tr.ewm(alpha=1/14, adjust=False).mean()
    df_atr = pd.DataFrame(atr_dict)
    
    # Macro Vol Z-Score
    spy_ret = close_p['SPY'].pct_change()
    ewma_lam = 1.0 - (2.0 / 21.0)
    spy_ewma_var = (spy_ret**2).ewm(alpha=(1 - ewma_lam), adjust=False).mean()
    spy_ewma_vol = np.sqrt(spy_ewma_var) * np.sqrt(252)
    spy_vol_ma = spy_ewma_vol.rolling(60).mean()
    spy_vol_std = spy_ewma_vol.rolling(60).std()
    spy_vol_zscore = (spy_ewma_vol - spy_vol_ma) / (spy_vol_std + 1e-9)
    
    return df_preds, close_p, df_atr, spy_vol_zscore, all_tickers

df_predictions, daily_prices, df_atr_matrix, macro_vol_z, universe_tickers = load_sweep_data()
all_sim_dates = sorted(list(set(df_predictions['date'].unique()) & set(daily_prices.index)))
print(f"✅ Loaded {len(df_predictions)} predictions over {len(all_sim_dates)} trading days across {len(universe_tickers)} tickers")

📈 Downloading OHLC market data for 101 tickers...


✅ Loaded 128799 predictions over 1295 trading days across 100 tickers


C:\Users\honza\AppData\Local\Temp\ipykernel_4180\3873613036.py:35: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  spy_ret = close_p['SPY'].pct_change()


## 3. Parametric Simulation Engines for Frequency Sweeps (1-120 Days)

In [3]:
def simulate_unified_one_for_all(
    preds_df, prices_df, atr_df, macro_z, all_dates,
    base_cap=0.08, max_confluence_cap=0.20, atr_multiplier=2.5, rebal_days=25, z_threshold=2.0, active_n=10, transaction_cost_bps=15
):
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_positions = {}
    history = []
    days_since = rebal_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        atr_now = atr_df.loc[d] if d in atr_df.index else None
        z_curr = macro_z.loc[d] if d in macro_z.index else 0.0
        
        stopped_out = []
        for t, pos in list(active_positions.items()):
            if t in p_now and pd.notna(p_now[t]):
                price_curr = p_now[t]
                atr_curr = atr_now[t] if (atr_now is not None and t in atr_now and pd.notna(atr_now[t])) else (price_curr * 0.03)
                if price_curr > pos['highest_price']:
                    pos['highest_price'] = price_curr
                    pos['stop_price'] = max(pos['stop_price'], price_curr - (atr_multiplier * atr_curr))
                if price_curr <= pos['stop_price']:
                    cash += pos['shares'] * price_curr * (1.0 - fee_rate)
                    stopped_out.append(t)
        for t in stopped_out:
            del active_positions[t]
            
        if days_since >= rebal_days:
            days_since = 0
            is_vol_spike = (pd.notna(z_curr) and z_curr > z_threshold)
            cash_buffer_ratio = 0.30 if is_vol_spike else 0.0
            
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds['predicted_return_multimodal'] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values('predicted_return_multimodal', ascending=False).head(active_n)
                conf_score = selected['confluence_score'] if 'confluence_score' in selected.columns else 0.0
                caps = base_cap + (conf_score / 6.0) * (max_confluence_cap - base_cap)
                caps = caps.clip(lower=base_cap, upper=max_confluence_cap)
                inv_vols = 1.0 / selected['ewma_volatility'].clip(lower=0.05)
                raw_weights = inv_vols / inv_vols.sum()
                bounded_weights = np.minimum(raw_weights, caps)
                final_weights = (bounded_weights / bounded_weights.sum()) * (1.0 - cash_buffer_ratio)
                target_alloc = dict(zip(selected['ticker'], final_weights))
            else:
                target_alloc = {}
                
            for t in list(active_positions.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_positions[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_positions[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now)
            cash = total_fund * cash_buffer_ratio
            investable = total_fund * (1.0 - cash_buffer_ratio)
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    price_curr = p_now[t]
                    atr_curr = atr_now[t] if (atr_now is not None and t in atr_now and pd.notna(atr_now[t])) else (price_curr * 0.03)
                    shares = (investable * (w / (1.0 - cash_buffer_ratio + 1e-9)) * (1.0 - fee_rate)) / price_curr
                    active_positions[t] = {
                        'shares': shares,
                        'entry_price': price_curr,
                        'highest_price': price_curr,
                        'stop_price': price_curr - (atr_multiplier * atr_curr),
                        'weight': w
                    }
                    
        days_since += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    df_h = pd.DataFrame(history)
    tot_ret = (df_h['portfolio_value'].iloc[-1] / df_h['portfolio_value'].iloc[0]) - 1.0
    return df_h, tot_ret

def simulate_active_strategy(
    preds_df, prices_df, all_dates,
    pred_col='predicted_return_multimodal',
    rebal_days=5, active_n=10, max_pos_cap=0.10, stop_loss_pct=0.10, transaction_cost_bps=15, use_risk_controls=True
):
    fee_rate = transaction_cost_bps / 10000.0
    portfolio_val = INITIAL_CAPITAL
    cash = INITIAL_CAPITAL
    active_pos = {}
    history = []
    days_since = rebal_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        
        stoppers = []
        for t, pos in list(active_pos.items()):
            if t in p_now and pd.notna(p_now[t]):
                unrel = (p_now[t] - pos['entry_price']) / pos['entry_price']
                if use_risk_controls and unrel <= -stop_loss_pct:
                    cash += pos['shares'] * p_now[t] * (1.0 - fee_rate)
                    stoppers.append(t)
        for t in stoppers:
            del active_pos[t]
            
        if days_since >= rebal_days:
            days_since = 0
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds[pred_col] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values(pred_col, ascending=False).head(active_n)
                if use_risk_controls and 'ewma_volatility' in selected.columns:
                    inv_vols = 1.0 / (selected['ewma_volatility'].clip(lower=0.05))
                    raw_weights = inv_vols / inv_vols.sum()
                    capped_weights = raw_weights.clip(upper=max_pos_cap)
                    final_weights = capped_weights / capped_weights.sum()
                else:
                    final_weights = [1.0 / len(selected)] * len(selected)
                target_alloc = dict(zip(selected['ticker'], final_weights))
            else:
                target_alloc = {}
                
            for t in list(active_pos.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_pos[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_pos[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_pos.items() if t in p_now)
            cash = 0.0
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    shares = (total_fund * w * (1.0 - fee_rate)) / p_now[t]
                    active_pos[t] = {'shares': shares, 'entry_price': p_now[t], 'weight': w}
                    
        days_since += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_pos.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    df_h = pd.DataFrame(history)
    tot_ret = (df_h['portfolio_value'].iloc[-1] / df_h['portfolio_value'].iloc[0]) - 1.0
    return df_h, tot_ret

def simulate_smart_bh_strategy(prices_df, all_dates, rebal_days=5, top_n=10, transaction_cost_bps=15):
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_pos = {}
    history = []
    days_since = rebal_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        
        if days_since >= rebal_days:
            days_since = 0
            mkt_vals = {t: p_now[t] for t in universe_tickers if t in p_now and pd.notna(p_now[t])}
            s_top = pd.Series(mkt_vals).sort_values(ascending=False).head(top_n)
            val_weights = s_top / s_top.sum()
            target_alloc = val_weights.to_dict()
            
            for t in list(active_pos.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_pos[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_pos[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_pos.items() if t in p_now)
            cash = 0.0
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    shares = (total_fund * w * (1.0 - fee_rate)) / p_now[t]
                    active_pos[t] = {'shares': shares, 'entry_price': p_now[t], 'weight': w}
                    
        days_since += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_pos.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    df_h = pd.DataFrame(history)
    tot_ret = (df_h['portfolio_value'].iloc[-1] / df_h['portfolio_value'].iloc[0]) - 1.0
    return df_h, tot_ret

## 4. Method 1: Unified "One-for-All" Alpha Engine (Flagship) Grid Sweep (1–120 Days / $M=100$)

In [4]:
print("🔄 Running Grid Sweep (1-120 Days) for Method 1: Unified 'One-for-All' Alpha Engine (M=100)...")
freq_range = list(range(1, 121))
m1_curves = {}
m1_profits = []

for k in tqdm(freq_range, desc="Unified Engine Sweep (M=100)"):
    df_curve, tot_ret = simulate_unified_one_for_all(
        df_predictions, daily_prices, df_atr_matrix, macro_vol_z, all_sim_dates,
        rebal_days=k, base_cap=0.08, max_confluence_cap=0.20, atr_multiplier=2.5, active_n=10
    )
    m1_curves[k] = df_curve
    m1_profits.append({'frequency_days': k, 'total_return_pct': tot_ret * 100.0})

df_m1_sweep = pd.DataFrame(m1_profits)
m1_top10 = df_m1_sweep.sort_values('total_return_pct', ascending=False).head(10)
print("=== METHOD 1: UNIFIED ENGINE TOP 10 OPTIMAL FREQUENCIES (M=100) ===")
m1_top10

🔄 Running Grid Sweep (1-120 Days) for Method 1: Unified 'One-for-All' Alpha Engine (M=100)...


Unified Engine Sweep (M=100):   0%|          | 0/120 [00:00<?, ?it/s]

=== METHOD 1: UNIFIED ENGINE TOP 10 OPTIMAL FREQUENCIES (M=100) ===


,frequency_days,total_return_pct
24,25,220.196551
34,35,212.899620
10,11,204.339159
6,7,189.598653
9,10,181.038080
7,8,176.730328
28,29,153.406950
15,16,151.770441
4,5,145.233229
18,19,135.771719


In [5]:
# Plot 1.1: Profit vs Frequency Curve (Method 1)
fig1_1 = px.line(
    df_m1_sweep, x='frequency_days', y='total_return_pct',
    title='<b>Method 1: Total Return (%) vs. Reallocation Frequency (1-120 Days / M=100)</b><br><sup>Unified "One-for-All" Alpha Engine (Flagship)</sup>',
    markers=True, template='plotly_dark'
)
fig1_1.update_traces(line_color='#00CC96', line_width=3)
fig1_1.update_layout(
    xaxis_title='Reallocation Frequency (Days)',
    yaxis_title='Total Return (%)',
    width=1100, height=480,
    margin=dict(l=60, r=40, t=80, b=60)
)
fig1_1.show()

# Plot 1.2: Top 10 Frequencies Equity Curves Time Series (Method 1)
fig1_2 = go.Figure()
palette = px.colors.qualitative.Plotly

for idx, row in enumerate(m1_top10.itertuples()):
    k = int(row.frequency_days)
    df_k = m1_curves[k]
    fig1_2.add_trace(go.Scatter(
        x=df_k['date'], y=df_k['portfolio_value'],
        name=f"{k}d (+{row.total_return_pct:.1f}%)",
        line=dict(width=3.0 if idx == 0 else 1.8, color=palette[idx % len(palette)])
    ))

fig1_2.update_layout(
    template='plotly_dark', width=1100, height=580,
    title='<b>Method 1: Top 10 Frequencies Portfolio Value Over Time (2021-2026 / M=100)</b><br><sup>Unified "One-for-All" Alpha Engine (Flagship)</sup>',
    xaxis_title='Date', yaxis_title='Portfolio Value ($)',
    margin=dict(l=60, r=180, t=80, b=60),
    legend=dict(
        orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02,
        title=dict(text='<b>Top 10 Cadences</b>')
    )
)
fig1_2.show()

## 5. Method 2: Multi-Modal Alpha (Active Top 10, Fixed 10% Cap) Grid Sweep (1–120 Days / $M=100$)

In [6]:
print("🔄 Running Grid Sweep (1-120 Days) for Method 2: Multi-Modal Alpha (Fixed 10% Cap, M=100)...")
m2_curves = {}
m2_profits = []

for k in tqdm(freq_range, desc="Multi-Modal Alpha Sweep (M=100)"):
    df_curve, tot_ret = simulate_active_strategy(
        df_predictions, daily_prices, all_sim_dates,
        pred_col='predicted_return_multimodal',
        rebal_days=k, active_n=10, max_pos_cap=0.10, stop_loss_pct=0.10, use_risk_controls=True
    )
    m2_curves[k] = df_curve
    m2_profits.append({'frequency_days': k, 'total_return_pct': tot_ret * 100.0})

df_m2_sweep = pd.DataFrame(m2_profits)
m2_top10 = df_m2_sweep.sort_values('total_return_pct', ascending=False).head(10)
print("=== METHOD 2: MULTI-MODAL TOP 10 OPTIMAL FREQUENCIES (M=100) ===")
m2_top10

🔄 Running Grid Sweep (1-120 Days) for Method 2: Multi-Modal Alpha (Fixed 10% Cap, M=100)...


Multi-Modal Alpha Sweep (M=100):   0%|          | 0/120 [00:00<?, ?it/s]

=== METHOD 2: MULTI-MODAL TOP 10 OPTIMAL FREQUENCIES (M=100) ===


,frequency_days,total_return_pct
24,25,289.128440
34,35,276.359595
56,57,258.148481
15,16,233.606289
6,7,227.827481
105,106,222.483162
18,19,219.058614
62,63,218.411538
7,8,217.310269
10,11,216.164301


In [7]:
# Plot 2.1: Profit vs Frequency Curve (Method 2)
fig2_1 = px.line(
    df_m2_sweep, x='frequency_days', y='total_return_pct',
    title='<b>Method 2: Total Return (%) vs. Reallocation Frequency (1-120 Days / M=100)</b><br><sup>Multi-Modal Alpha (Active Top 10, Fixed 10% Cap)</sup>',
    markers=True, template='plotly_dark'
)
fig2_1.update_traces(line_color='#00B4D8', line_width=3)
fig2_1.update_layout(
    xaxis_title='Reallocation Frequency (Days)',
    yaxis_title='Total Return (%)',
    width=1100, height=480,
    margin=dict(l=60, r=40, t=80, b=60)
)
fig2_1.show()

# Plot 2.2: Top 10 Frequencies Equity Curves Time Series (Method 2)
fig2_2 = go.Figure()

for idx, row in enumerate(m2_top10.itertuples()):
    k = int(row.frequency_days)
    df_k = m2_curves[k]
    fig2_2.add_trace(go.Scatter(
        x=df_k['date'], y=df_k['portfolio_value'],
        name=f"{k}d (+{row.total_return_pct:.1f}%)",
        line=dict(width=3.0 if idx == 0 else 1.8, color=palette[idx % len(palette)])
    ))

fig2_2.update_layout(
    template='plotly_dark', width=1100, height=580,
    title='<b>Method 2: Top 10 Frequencies Portfolio Value Over Time (2021-2026 / M=100)</b><br><sup>Multi-Modal Alpha (Active Top 10, Fixed 10% Cap)</sup>',
    xaxis_title='Date', yaxis_title='Portfolio Value ($)',
    margin=dict(l=60, r=180, t=80, b=60),
    legend=dict(
        orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02,
        title=dict(text='<b>Top 10 Cadences</b>')
    )
)
fig2_2.show()

## 6. Method 3: Baseline Naive XGBoost (Top 10 Active) Grid Sweep (1–120 Days / $M=100$)

In [8]:
print("🔄 Running Grid Sweep (1-120 Days) for Method 3: Baseline Naive XGBoost (M=100)...")
m3_curves = {}
m3_profits = []

for k in tqdm(freq_range, desc="Baseline XGBoost Sweep (M=100)"):
    df_curve, tot_ret = simulate_active_strategy(
        df_predictions, daily_prices, all_sim_dates,
        pred_col='predicted_return_baseline',
        rebal_days=k, active_n=10, max_pos_cap=1.0, stop_loss_pct=1.0, use_risk_controls=False
    )
    m3_curves[k] = df_curve
    m3_profits.append({'frequency_days': k, 'total_return_pct': tot_ret * 100.0})

df_m3_sweep = pd.DataFrame(m3_profits)
m3_top10 = df_m3_sweep.sort_values('total_return_pct', ascending=False).head(10)
print("=== METHOD 3: BASELINE XGBOOST TOP 10 OPTIMAL FREQUENCIES (M=100) ===")
m3_top10

🔄 Running Grid Sweep (1-120 Days) for Method 3: Baseline Naive XGBoost (M=100)...


Baseline XGBoost Sweep (M=100):   0%|          | 0/120 [00:00<?, ?it/s]

=== METHOD 3: BASELINE XGBOOST TOP 10 OPTIMAL FREQUENCIES (M=100) ===


,frequency_days,total_return_pct
105,106,438.729396
54,55,431.196510
95,96,387.478912
115,116,387.237963
42,43,372.647326
18,19,370.021565
102,103,369.973499
72,73,367.615352
104,105,366.881273
56,57,361.472076


In [9]:
# Plot 3.1: Profit vs Frequency Curve (Method 3)
fig3_1 = px.line(
    df_m3_sweep, x='frequency_days', y='total_return_pct',
    title='<b>Method 3: Total Return (%) vs. Reallocation Frequency (1-120 Days / M=100)</b><br><sup>Baseline Naive XGBoost (Top 10 Active)</sup>',
    markers=True, template='plotly_dark'
)
fig3_1.update_traces(line_color='#AB63FA', line_width=3)
fig3_1.update_layout(
    xaxis_title='Reallocation Frequency (Days)',
    yaxis_title='Total Return (%)',
    width=1100, height=480,
    margin=dict(l=60, r=40, t=80, b=60)
)
fig3_1.show()

# Plot 3.2: Top 10 Frequencies Equity Curves Time Series (Method 3)
fig3_2 = go.Figure()

for idx, row in enumerate(m3_top10.itertuples()):
    k = int(row.frequency_days)
    df_k = m3_curves[k]
    fig3_2.add_trace(go.Scatter(
        x=df_k['date'], y=df_k['portfolio_value'],
        name=f"{k}d (+{row.total_return_pct:.1f}%)",
        line=dict(width=3.0 if idx == 0 else 1.8, color=palette[idx % len(palette)])
    ))

fig3_2.update_layout(
    template='plotly_dark', width=1100, height=580,
    title='<b>Method 3: Top 10 Frequencies Portfolio Value Over Time (2021-2026 / M=100)</b><br><sup>Baseline Naive XGBoost (Top 10 Active)</sup>',
    xaxis_title='Date', yaxis_title='Portfolio Value ($)',
    margin=dict(l=60, r=180, t=80, b=60),
    legend=dict(
        orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02,
        title=dict(text='<b>Top 10 Cadences</b>')
    )
)
fig3_2.show()

## 7. Method 4: Smart Buy & Hold (Top 10 Market-Cap Value-Weighted) Grid Sweep (1–120 Days / $M=100$)

In [10]:
print("🔄 Running Grid Sweep (1-120 Days) for Method 4: Smart Buy & Hold (M=100)...")
m4_curves = {}
m4_profits = []

for k in tqdm(freq_range, desc="Smart B&H Sweep (M=100)"):
    df_curve, tot_ret = simulate_smart_bh_strategy(
        daily_prices, all_sim_dates, rebal_days=k, top_n=10, transaction_cost_bps=15
    )
    m4_curves[k] = df_curve
    m4_profits.append({'frequency_days': k, 'total_return_pct': tot_ret * 100.0})

df_m4_sweep = pd.DataFrame(m4_profits)
m4_top10 = df_m4_sweep.sort_values('total_return_pct', ascending=False).head(10)
print("=== METHOD 4: SMART BUY & HOLD TOP 10 OPTIMAL FREQUENCIES (M=100) ===")
m4_top10

🔄 Running Grid Sweep (1-120 Days) for Method 4: Smart Buy & Hold (M=100)...


Smart B&H Sweep (M=100):   0%|          | 0/120 [00:00<?, ?it/s]

=== METHOD 4: SMART BUY & HOLD TOP 10 OPTIMAL FREQUENCIES (M=100) ===


,frequency_days,total_return_pct
70,71,56.865179
71,72,52.545770
79,80,49.271412
80,81,48.479607
91,92,47.220372
73,74,46.025751
100,101,46.008005
67,68,45.925322
119,120,45.256840
78,79,44.963066


In [11]:
# Plot 4.1: Profit vs Frequency Curve (Method 4)
fig4_1 = px.line(
    df_m4_sweep, x='frequency_days', y='total_return_pct',
    title='<b>Method 4: Total Return (%) vs. Reallocation Frequency (1-120 Days / M=100)</b><br><sup>Smart Buy & Hold (Top 10 Market-Cap Value-Weighted)</sup>',
    markers=True, template='plotly_dark'
)
fig4_1.update_traces(line_color='#FFA15A', line_width=3)
fig4_1.update_layout(
    xaxis_title='Reallocation Frequency (Days)',
    yaxis_title='Total Return (%)',
    width=1100, height=480,
    margin=dict(l=60, r=40, t=80, b=60)
)
fig4_1.show()

# Plot 4.2: Top 10 Frequencies Equity Curves Time Series (Method 4)
fig4_2 = go.Figure()

for idx, row in enumerate(m4_top10.itertuples()):
    k = int(row.frequency_days)
    df_k = m4_curves[k]
    fig4_2.add_trace(go.Scatter(
        x=df_k['date'], y=df_k['portfolio_value'],
        name=f"{k}d (+{row.total_return_pct:.1f}%)",
        line=dict(width=3.0 if idx == 0 else 1.8, color=palette[idx % len(palette)])
    ))

fig4_2.update_layout(
    template='plotly_dark', width=1100, height=580,
    title='<b>Method 4: Top 10 Frequencies Portfolio Value Over Time (2021-2026 / M=100)</b><br><sup>Smart Buy & Hold (Top 10 Market-Cap Value-Weighted)</sup>',
    xaxis_title='Date', yaxis_title='Portfolio Value ($)',
    margin=dict(l=60, r=180, t=80, b=60),
    legend=dict(
        orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02,
        title=dict(text='<b>Top 10 Cadences</b>')
    )
)
fig4_2.show()

## 8. Cross-Strategy Spectrum Comparison & Export Results ($M=100$)

In [12]:
df_all_sweeps = pd.DataFrame({
    'frequency_days': freq_range,
    'Unified_OneForAll_Return_Pct': df_m1_sweep['total_return_pct'].values,
    'MultiModal_Alpha_Return_Pct': df_m2_sweep['total_return_pct'].values,
    'Baseline_XGBoost_Return_Pct': df_m3_sweep['total_return_pct'].values,
    'Smart_BuyHold_Return_Pct': df_m4_sweep['total_return_pct'].values
})

# Cross-method frequency spectrum comparison plot
fig_comp = go.Figure()
fig_comp.add_trace(go.Scatter(x=df_all_sweeps['frequency_days'], y=df_all_sweeps['Unified_OneForAll_Return_Pct'], name='Unified "One-for-All" Engine (Flagship)', line=dict(color='#00CC96', width=3.5)))
fig_comp.add_trace(go.Scatter(x=df_all_sweeps['frequency_days'], y=df_all_sweeps['MultiModal_Alpha_Return_Pct'], name='Multi-Modal Alpha (Fixed 10% Cap)', line=dict(color='#00B4D8', width=2)))
fig_comp.add_trace(go.Scatter(x=df_all_sweeps['frequency_days'], y=df_all_sweeps['Baseline_XGBoost_Return_Pct'], name='Baseline Naive XGBoost', line=dict(color='#AB63FA', width=2)))
fig_comp.add_trace(go.Scatter(x=df_all_sweeps['frequency_days'], y=df_all_sweeps['Smart_BuyHold_Return_Pct'], name='Smart Buy & Hold', line=dict(color='#FFA15A', width=2)))

fig_comp.update_layout(
    template='plotly_dark', width=1100, height=550,
    title='<b>Cross-Strategy Reallocation Frequency Sensitivity Spectrum (1-120 Days / M=100)</b>',
    xaxis_title='Reallocation Frequency (Days)', yaxis_title='Total Return (%)',
    margin=dict(l=60, r=180, t=80, b=60),
    legend=dict(
        orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02,
        title=dict(text='<b>Strategy</b>')
    )
)
fig_comp.show()

output_sweep_path = os.path.join(LOCAL_DATA_DIR, "rebalance_frequency_sweeps_poc.xlsx")
with pd.ExcelWriter(output_sweep_path) as writer:
    df_all_sweeps.to_excel(writer, sheet_name='all_frequencies_1_to_120d', index=False)
    m1_top10.to_excel(writer, sheet_name='unified_engine_top10_freqs', index=False)
    m2_top10.to_excel(writer, sheet_name='multimodal_top10_freqs', index=False)
    m3_top10.to_excel(writer, sheet_name='baseline_top10_freqs', index=False)
    m4_top10.to_excel(writer, sheet_name='smart_bh_top10_freqs', index=False)

print(f"💾 Successfully exported 1-120d frequency sweeps (M=100) to: {output_sweep_path}")

💾 Successfully exported 1-120d frequency sweeps (M=100) to: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched\rebalance_frequency_sweeps_poc.xlsx
